# 02 · ข้อ 1 วันธรรมดา vs วันหยุดสุดสัปดาห์
ผู้รับผิดชอบ: สมาชิก A · ต้องรัน `01_cleaning.ipynb` ก่อน

**คำถาม:** ปริมาณการใช้บริการขนส่งแตกต่างกันอย่างไรระหว่างวันธรรมดาและวันหยุดสุดสัปดาห์ ต่างกันอย่างไรในแต่ละวันของสัปดาห์ และรูปแบบนี้เปลี่ยนไปในแต่ละเดือนหรือไม่

โครงของ notebook ตรงกับหัวข้อที่รายงานต้องมีในแต่ละข้อ: Feature Selection → Data Preparation → Feature Engineering → Visualization → Analysis & Findings

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import prep

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
prep.setup_thai_font()

'Loma'

In [2]:
df = prep.load_clean()
catalog = pd.read_csv(prep.PREPARED_DIR / "service_catalog.csv", encoding="utf-8-sig", keep_default_na=False)
print(df.shape, df["year_be"].value_counts().to_dict())

(18705, 18) {2568: 15695, 2569: 3010}


## 1. Feature Selection

| Feature ต้นฉบับ | เหตุผล |
|---|---|
| `date` | สร้างวันในสัปดาห์และเดือน |
| `service`, `purpose`, `mode` | เปรียบเทียบรูปแบบระหว่างบริการ/กลุ่มการเดินทาง |
| `value` | จำนวนผู้ใช้บริการต่อวัน |
| `unit`, `exclude_reason`, `is_aggregate` | ใช้กรอง (คน เท่านั้น, ไม่ใช่ยอดรวม) |

## 2. Data Preparation

In [3]:
# ปี 2568 (ครบปี) · หน่วย "คน" · ตัดค่าว่างและศูนย์ช่วงหยุดรายงาน (ไม่นับเป็น 0)
q1 = prep.passenger_rows(df)
q1 = q1[(q1["year_be"] == 2568) & (~q1["missing_flag"]) & (~q1["zero_long_run"])].copy()
print(q1.shape, q1["service"].nunique(), "services")
print("แถวที่ตัดเพราะค่าว่าง/ศูนย์ช่วงยาว:",
      len(prep.passenger_rows(df).query("year_be == 2568")) - len(q1))

(13664, 18) 38 services
แถวที่ตัดเพราะค่าว่าง/ศูนย์ช่วงยาว: 206


## 3. Feature Engineering (อย่างน้อย 3 ตัวแปร)

| Feature ใหม่ | สูตร / เงื่อนไข | ประโยชน์ |
|---|---|---|
| `day_of_week`, `month` | `date.dt.dayofweek` (0 = จันทร์), `date.dt.month` | แกนจัดกลุ่ม |
| `is_weekend` | `day_of_week in {5, 6}` | แบ่งกลุ่มหลักตามโจทย์ |
| `dow_index` | `value / mean(value ของบริการนั้นทั้งปี) × 100` | เทียบบริการต่างขนาดบนสเกลเดียว |
| `weekend_ratio` | `mean(weekend) / mean(weekday)` ต่อบริการ × เดือน | ตัวชี้วัดเดียว: <1 = เดินทางไปทำงาน, >1 = ท่องเที่ยว |
| `pct_diff_weekend` | `(weekend_ratio − 1) × 100` | อ่านง่ายสำหรับสไลด์ |
| `is_public_holiday` (ส่วนขยาย) | เทียบปฏิทินวันหยุดราชการ 2568 | แยกวันหยุดนักขัตฤกษ์ที่ตรงกับวันธรรมดา |

In [4]:
# TODO: สร้าง features ตามตารางด้านบน แล้วแสดง .head() และ .describe() เป็นหลักฐาน

## 4. Visualization (อย่างน้อย 2 รูปแบบ)
แนะนำ: ① bar/line ของ `dow_index` จันทร์→อาทิตย์ แยกตาม `purpose` ② heatmap เดือน × วันในสัปดาห์ ③ line ของ `weekend_ratio` รายเดือน

บันทึกทุกกราฟด้วย `prep.save_fig(fig, "<ชื่อ>")` แล้วใช้ไฟล์ใน `figures/` ในรายงานและสไลด์

In [5]:
# TODO: กราฟที่ 1

**การตีความกราฟที่ 1:** _(เขียนจากตัวเลขจริงในกราฟ)_

In [6]:
# TODO: กราฟที่ 2

**การตีความกราฟที่ 2:** _(เขียนจากตัวเลขจริงในกราฟ)_

## 5. Analysis & Findings
- คำตอบของคำถาม:
- ข้อค้นพบสำคัญ:
- ข้อจำกัดของข้อมูล:

## 6. บันทึก Prepared Dataset

In [7]:
# TODO: เปลี่ยน result_df เป็นตารางสุดท้ายที่มี features ใหม่ครบ
# result_df.to_csv(prep.PREPARED_DIR / "q1_daily_features.csv", index=False, encoding="utf-8-sig")